In [1]:
import pandas as pd
import numpy as np

# Загружаем датасет
df = pd.read_csv('hotel_bookings.csv')

# Смотрим базовую информацию
print(f"Размер датасета: {df.shape}")
display(df.head(3))
display(df.info())


Размер датасета: (119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

None

In [2]:
# Удаляем признаки-утечки и ненужные колонки, о которых говорится в задании
# reservation_status, reservation_status_date содержат факт отмены/выезда в явном виде
# company часто пустая или дает прямое знание о корпоративном клиенте из будущего
cols_to_drop = ['reservation_status', 'reservation_status_date', 'company']
df = df.drop(columns=cols_to_drop, errors='ignore')

# Проверяем баланс классов целевой переменной (is_canceled)
print("\nБаланс классов (is_canceled):")
print(df['is_canceled'].value_counts(normalize=True))



Баланс классов (is_canceled):
is_canceled
0    0.629584
1    0.370416
Name: proportion, dtype: float64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Выделяем признаки (X) и целевую переменную (y)
X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

# 2. Делим на train и test (согласно условиям Блока А: test_size=0.2, stratify=y, random_state=42)[cite: 2]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Определяем типы колонок
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = ['hotel', 'meal', 'country', 'market_segment', 'deposit_type', 'customer_type']
categorical_features = [col for col in categorical_features if col in X.columns]

# 4. Трансформеры для чисел и категорий
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# 5. Итоговый пайплайн с логистической регрессией
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 6. Обучение и предсказание
print("Обучаем модель...")
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print("\nРезультаты базовой модели:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Обучаем модель...


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

# 1. Выделяем признаки и целевую переменную
X = df.drop(columns=['is_canceled'])
y = df['is_canceled']

# 2. Делим на train и тест
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 3. Определяем типы колонок
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = ['hotel', 'meal', 'country', 'market_segment', 'deposit_type', 'customer_type']
categorical_features = [col for col in categorical_features if col in X.columns]

# 4. Трансформеры (добавили импутер для чисел, заполняем нулями по условию)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

# 5. Итоговый пайплайн
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# 6. Обучение и предсказание
print("Обучаем модель...")
model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)

print("\nРезультаты базовой модели:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Обучаем модель...

Результаты базовой модели:
Accuracy: 0.8102

Confusion Matrix:
[[13633  1400]
 [ 3133  5712]]


In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Настраиваем кросс-валидацию (по условию Блока Б: StratifiedKFold(5))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Считаем метрику accuracy на кросс-валидации
cv_scores = cross_val_score(model_pipeline, X, y, cv=cv, scoring='accuracy')

print("Результаты кросс-валидации (StratifiedKFold на 5 фолдах):")
print(f"Accuracy на каждом фолде: {cv_scores}")
print(f"Средняя Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")

Результаты кросс-валидации (StratifiedKFold на 5 фолдах):
Accuracy на каждом фолде: [0.80865232 0.81313343 0.81141637 0.81007622 0.81317531]
Средняя Accuracy: 0.8113 (±0.0018)


In [6]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Обучаем модель заново на train
model_pipeline.fit(X_train, y_train)

# Получаем вероятности классов для тестовой выборки
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

# Попробуем разные пороги (thresholds), чтобы найти оптимальный под задачи отеля
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

print("Сравнение разных порогов классификации:")
print("-" * 50)
for th in thresholds:
    y_pred_th = (y_proba >= th).astype(int)
    print(f"Порог: {th}")
    print(f"  Precision: {precision_score(y_test, y_pred_th):.4f}")
    print(f"  Recall (полнота отмен): {recall_score(y_test, y_pred_th):.4f}")
    print(f"  F1-score: {f1_score(y_test, y_pred_th):.4f}")
    print("-" * 50)

Сравнение разных порогов классификации:
--------------------------------------------------
Порог: 0.3
  Precision: 0.6738
  Recall (полнота отмен): 0.8252
  F1-score: 0.7419
--------------------------------------------------
Порог: 0.4
  Precision: 0.7464
  Recall (полнота отмен): 0.7323
  F1-score: 0.7393
--------------------------------------------------
Порог: 0.5
  Precision: 0.8031
  Recall (полнота отмен): 0.6458
  F1-score: 0.7159
--------------------------------------------------
Порог: 0.6
  Precision: 0.8654
  Recall (полнота отмен): 0.5605
  F1-score: 0.6804
--------------------------------------------------
Порог: 0.7
  Precision: 0.9145
  Recall (полнота отмен): 0.4876
  F1-score: 0.6361
--------------------------------------------------


In [7]:
from sklearn.metrics import classification_report

# 1. Сравнение с моделью class_weight='balanced'
model_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

model_balanced.fit(X_train, y_train)
y_pred_bal = model_balanced.predict(X_test)

print("Результаты модели с class_weight='balanced':")
print(confusion_matrix(y_test, y_pred_bal))
print("\nОтчет о классификации:")
print(classification_report(y_test, y_pred_bal))

Результаты модели с class_weight='balanced':
[[12477  2556]
 [ 2148  6697]]

Отчет о классификации:
              precision    recall  f1-score   support

           0       0.85      0.83      0.84     15033
           1       0.72      0.76      0.74      8845

    accuracy                           0.80     23878
   macro avg       0.79      0.79      0.79     23878
weighted avg       0.81      0.80      0.80     23878



In [8]:
# Находим индексы, где сбалансированная модель ошиблась
errors = (y_test != y_pred_bal)
X_err = X_test[errors].copy()
X_err['true_value'] = y_test[errors]
X_err['pred_value'] = y_pred_bal[errors]

print(f"Всего ошибок на тестовой выборке: {errors.sum()} из {len(y_test)}")

# Посмотрим, в каких типах депозита или сегмента рынка ошибки происходят чаще всего
print("\nОшибки в разрезе типа депозита (deposit_type):")
print(X_err['deposit_type'].value_counts())

print("\nОшибки в разрезе типа клиента (customer_type):")
print(X_err['customer_type'].value_counts())

Всего ошибок на тестовой выборке: 4704 из 23878

Ошибки в разрезе типа депозита (deposit_type):
deposit_type
No Deposit    4683
Non Refund      18
Refundable       3
Name: count, dtype: int64

Ошибки в разрезе типа клиента (customer_type):
customer_type
Transient          3773
Transient-Party     860
Contract             64
Group                 7
Name: count, dtype: int64
